# LeetCode 3683 — Earliest Time to Finish One Task bằng Python

Link đề: https://leetcode.com/problems/earliest-time-to-finish-one-task/ · Độ khó: **Easy** · Chủ đề: Array, Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def earliestTime(self, tasks: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi công việc là cặp `[s, t]` (bắt đầu lúc `s`, kéo dài `t`), thời điểm **kết thúc** là `s + t`; cần tìm thời điểm **sớm nhất** mà **ít nhất một** công việc đã xong.
2. Biết duyệt danh sách các cặp bằng `for s, t in tasks` (**unpacking**) và dùng `min` với biểu thức sinh.
3. Phân biệt rõ **ba đại lượng**: thời điểm bắt đầu, thời lượng, thời điểm kết thúc; và hiểu vì sao `min(s) + min(t)` **không** là đáp án.
4. Viết 4 cách: vòng lặp theo dõi giá trị nhỏ nhất, `min` một dòng, sắp xếp theo thời điểm kết thúc, và tìm kiếm nhị phân theo đáp án; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: lấy `min` của thời điểm bắt đầu, `min` của thời lượng, `min(s) + min(t)`, và dùng `max` thay `min`.

## 2. Đề bài & ví dụ

Cho mảng hai chiều `tasks`, trong đó `tasks[i] = [s_i, t_i]` nghĩa là công việc `i` **bắt đầu** ở thời điểm `s_i` và cần `t_i` đơn vị thời gian để hoàn thành (nên **kết thúc** ở `s_i + t_i`). Trả về **thời điểm sớm nhất** mà **ít nhất một** công việc đã hoàn thành.

| Ví dụ | tasks | Output | Giải thích |
|---|---|---|---|
| 1 | `[[1,6],[2,3]]` | `5` | công việc 1 kết thúc lúc `1+6 = 7`; công việc 2 kết thúc lúc `2+3 = 5`; sớm nhất là `5` |
| 2 | `[[100,100],[100,100],[100,100]]` | `200` | cả ba cùng kết thúc lúc `200` |

Constraints: `1 <= tasks.length <= 100`, `tasks[i] = [s_i, t_i]`, `1 <= s_i, t_i <= 100`.

In [1]:
tasks = [[1, 6], [2, 3]]
for s, t in tasks:
    print(f"bắt đầu {s}, kéo dài {t} -> kết thúc {s + t}")
print("sớm nhất =", min(s + t for s, t in tasks))

bắt đầu 1, kéo dài 6 -> kết thúc 7
bắt đầu 2, kéo dài 3 -> kết thúc 5
sớm nhất = 5


## 3. Kiến thức nền

### 3.1. Danh sách các cặp và unpacking

`tasks` là list chứa các list nhỏ `[s, t]`. Vòng `for s, t in tasks` **giải nén** (unpack) mỗi cặp thành hai biến. Tương đương `for task in tasks: s, t = task[0], task[1]`.

In [2]:
tasks = [[1, 6], [2, 3]]
for task in tasks:
    print(task, task[0], task[1])
for s, t in tasks:
    print(s, t)
a, b = [7, 8]
print(a, b)

[1, 6] 1 6
[2, 3] 2 3
1 6
2 3
7 8


### 3.2. `min` với biểu thức sinh

`min(iterable)` trả phần tử nhỏ nhất. Với biểu thức sinh `min(s + t for s, t in tasks)` Python tính `s + t` cho từng cặp rồi lấy nhỏ nhất, không cần tạo danh sách trung gian.

In [3]:
tasks = [[1, 6], [2, 3]]
print(min(s + t for s, t in tasks))
print(min([7, 5, 9]), max([7, 5, 9]))
print(min(tasks))                    # min trên list: so sánh theo phần tử đầu rồi phần tử sau

5
5 9
[1, 6]


### 3.3. Sắp xếp theo khóa (`key`)

`sorted(tasks, key=lambda p: p[0] + p[1])` sắp các cặp theo thời điểm kết thúc tăng dần; phần tử đầu tiên là công việc xong sớm nhất. Chú ý: chỉ cần phần tử nhỏ nhất nên sắp xếp là **thừa** (`O(n log n)` thay vì `O(n)`).

In [4]:
tasks = [[1, 6], [2, 3], [4, 0 + 1]]
ordered = sorted(tasks, key=lambda p: p[0] + p[1])
print(ordered)
print(ordered[0][0] + ordered[0][1])

[[2, 3], [4, 1], [1, 6]]
5


### 3.4. Tìm kiếm nhị phân theo đáp án

Hàm `feasible(T)`: "đến thời điểm `T` đã có ít nhất một công việc xong?" là **đơn điệu** (đúng với `T` thì đúng với `T + 1`). Tìm `T` **nhỏ nhất** đúng bằng nhị phân. Đây là mẫu "nhị phân theo đáp án": nhớ `lo < hi`, `mid = (lo + hi) // 2`, đúng thì `hi = mid`, sai thì `lo = mid + 1`.

In [5]:
tasks = [[1, 6], [2, 3]]
def feasible(T):
    return any(s + t <= T for s, t in tasks)

lo, hi = 0, 300
while lo < hi:
    mid = (lo + hi) // 2
    if feasible(mid):
        hi = mid
    else:
        lo = mid + 1
print(lo)

5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `[[1,6],[2,3]]`.

```
công việc 1: bắt đầu 1, dài 6 -> kết thúc 7
công việc 2: bắt đầu 2, dài 3 -> kết thúc 5
"ít nhất một công việc xong" -> chọn công việc xong sớm nhất -> 5
```

Chú ý đừng nhầm: công việc 1 bắt đầu **sớm hơn** (`1 < 2`) nhưng dài hơn nên **kết thúc muộn hơn**. Cái cần so sánh là **thời điểm kết thúc**, tức `s + t`.

### Mô phỏng dòng thời gian

Để "thấy" vì sao đáp án là min, ta cho đồng hồ chạy `T = 0, 1, 2, ...` và liệt kê công việc đã xong tại mỗi thời điểm. Lần đầu tiên danh sách khác rỗng chính là đáp án. Đồng hồ chạy tới tối đa `200` nên rất rẻ với `n <= 100`.

In [6]:
tasks = [[4, 2], [1, 9], [3, 1], [6, 6]]
for T in range(0, 8):
    done = [i for i, (s, t) in enumerate(tasks) if s + t <= T]
    print(f"T={T}: việc đã xong = {done}")
    if done:
        print("-> lần đầu tiên có việc xong lúc", T)
        break

T=0: việc đã xong = []
T=1: việc đã xong = []
T=2: việc đã xong = []
T=3: việc đã xong = []
T=4: việc đã xong = [2]
-> lần đầu tiên có việc xong lúc 4


## 5. Cách 1 — Brute Force / baseline

Duyệt từng công việc, tính `end = s + t`, giữ giá trị nhỏ nhất đã gặp (khởi tạo bằng một số rất lớn hoặc bằng công việc đầu tiên).

Nút thắt: không có — bài chỉ cần một lượt duyệt `O(n)`. Điểm cần cẩn thận duy nhất là khởi tạo và đúng khóa `s + t`.

In [7]:
class SolutionLoop:
    def earliestTime(self, tasks):
        best = tasks[0][0] + tasks[0][1]
        for i in range(1, len(tasks)):
            end = tasks[i][0] + tasks[i][1]
            if end < best:
                best = end
        return best

print(SolutionLoop().earliestTime([[1, 6], [2, 3]]))
print(SolutionLoop().earliestTime([[100, 100], [100, 100], [100, 100]]))

5
200


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** "ít nhất một công việc xong" đồng nghĩa với "công việc xong **sớm nhất**". Mỗi công việc độc lập: nó kết thúc đúng lúc `s + t`, không bị ảnh hưởng bởi công việc khác (các công việc chạy song song). Vậy đáp án là **`min` của `s + t`**.

**Quy tắc "nếu… thì…":**

- Mỗi công việc có thời điểm kết thúc `s + t`.
- Thời điểm "đã có việc xong" chính là min các thời điểm kết thúc.
- Không cần xét thứ tự, không cần sắp xếp.

Invariant của vòng lặp: sau khi duyệt `i` công việc, `best` là thời điểm kết thúc nhỏ nhất trong `i` công việc đó.

In [8]:
def earliest(tasks):
    ends = [s + t for s, t in tasks]
    print("thời điểm kết thúc:", ends)
    return min(ends)

print(earliest([[1, 6], [2, 3]]))
print(earliest([[5, 5]]))

thời điểm kết thúc: [7, 5]
5
thời điểm kết thúc: [10]
10


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) vòng lặp theo dõi `best`; (2) `min` một dòng; (3) sắp xếp theo thời điểm kết thúc; (4) nhị phân theo đáp án.

### Cách 2 — `min` một dòng

Biểu thức sinh `s + t` rồi `min`.

In [9]:
class SolutionMin:
    def earliestTime(self, tasks):
        return min(s + t for s, t in tasks)

print(SolutionMin().earliestTime([[1, 6], [2, 3]]))
print(SolutionMin().earliestTime([[100, 100], [100, 100], [100, 100]]))

5
200


Độ phức tạp Cách 2: Time `O(n)`, Space `O(1)`.

### Cách 3 — Sắp xếp theo thời điểm kết thúc

Sắp `tasks` theo khóa `s + t` rồi lấy phần tử đầu. Đúng nhưng thừa (`O(n log n)`); hữu ích khi còn cần thứ tự các công việc xong tiếp theo (ví dụ hỏi công việc xong **thứ `k`**).

In [10]:
class SolutionSort:
    def earliestTime(self, tasks):
        ordered = sorted(tasks, key=lambda p: p[0] + p[1])
        first = ordered[0]
        return first[0] + first[1]

print(SolutionSort().earliestTime([[1, 6], [2, 3]]))
print(SolutionSort().earliestTime([[100, 100], [100, 100], [100, 100]]))

5
200


Độ phức tạp Cách 3: Time `O(n log n)`, Space `O(n)`.

### Cách 4 — Nhị phân theo đáp án

Đáp án nằm trong `[2, 200]` (`s, t` trong `1..100`). `feasible(T) = any(s + t <= T)`. Nhị phân tìm `T` nhỏ nhất đúng. Mỗi lần kiểm tra `O(n)`, tổng `O(n log 200)`: chậm hơn cách 2 nhưng là mẫu hay.

In [11]:
class SolutionBinary:
    def earliestTime(self, tasks):
        def feasible(T):
            return any(s + t <= T for s, t in tasks)

        lo, hi = 2, 200
        while lo < hi:
            mid = (lo + hi) // 2
            if feasible(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

print(SolutionBinary().earliestTime([[1, 6], [2, 3]]))
print(SolutionBinary().earliestTime([[100, 100], [100, 100], [100, 100]]))

5
200


Độ phức tạp Cách 4: Time `O(n log V)` với `V = 200`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 1 trên `tasks = [[4,2],[1,9],[3,1],[6,6]]`:

| bước | công việc | `end = s + t` | `best` trước | `best` sau |
|---|---|---|---|---|
| 0 | `[4,2]` | 6 | (khởi tạo) | 6 |
| 1 | `[1,9]` | 10 | 6 | 6 |
| 2 | `[3,1]` | 4 | 6 | 4 |
| 3 | `[6,6]` | 12 | 4 | 4 |

Đáp án `4`.

In [12]:
def trace_earliest(tasks):
    best = tasks[0][0] + tasks[0][1]
    print(f"khởi tạo: công việc {tasks[0]} -> best = {best}")
    for i in range(1, len(tasks)):
        s, t = tasks[i]
        end = s + t
        new_best = min(best, end)
        print(f"i={i}: công việc {tasks[i]} -> end = {end} | best {best} -> {new_best}")
        best = new_best
    print("đáp án =", best)
    return best

trace_earliest([[4, 2], [1, 9], [3, 1], [6, 6]])
trace_earliest([[1, 6], [2, 3]])

khởi tạo: công việc [4, 2] -> best = 6
i=1: công việc [1, 9] -> end = 10 | best 6 -> 6
i=2: công việc [3, 1] -> end = 4 | best 6 -> 4
i=3: công việc [6, 6] -> end = 12 | best 4 -> 4
đáp án = 4
khởi tạo: công việc [1, 6] -> best = 7
i=1: công việc [2, 3] -> end = 5 | best 7 -> 5
đáp án = 5


5

## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def earliestTime(self, tasks: List[List[int]]) -> int:
        return min(s + t for s, t in tasks)

print(Solution().earliestTime([[1, 6], [2, 3]]))
print(Solution().earliestTime([[100, 100], [100, 100], [100, 100]]))

5
200


## 10. Vì sao thuật toán đúng

- **Độc lập giữa các công việc:** công việc `i` kết thúc đúng lúc `s_i + t_i` bất kể công việc khác; không có ràng buộc "một lần chỉ làm một việc".
- **Đáp án = min:** tại thời điểm `T`, tập công việc đã xong là `{i : s_i + t_i <= T}`. Điều kiện "có ít nhất một" đúng khi `T >= min_i (s_i + t_i)`. Vậy thời điểm sớm nhất là chính giá trị min đó.
- **Cách 1:** theo invariant, sau mỗi bước `best = min` các `end` đã duyệt; sau khi duyệt hết là min toàn cục.
- **Cách 3:** sau khi sắp theo `s + t`, phần tử đầu có khóa nhỏ nhất, đúng bằng min.
- **Cách 4:** `feasible` đơn điệu nên nhị phân hội tụ về min; kết quả bằng min vì `feasible(T)` đúng đầu tiên tại `T = min`.
- **Biên:** `n >= 1` nên luôn có ít nhất một công việc; không cần trả giá trị đặc biệt cho mảng rỗng.

## 11. Phân tích Time / Space Complexity

`n = len(tasks) <= 100`, `V = 200` là cận trên của đáp án.

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp theo dõi `best` | `O(n)` | `O(1)` |
| 2. `min(s + t ...)` | `O(n)` | `O(1)` |
| 3. Sắp xếp theo `s + t` | `O(n log n)` | `O(n)` |
| 4. Nhị phân theo đáp án | `O(n log V)` | `O(1)` |

Cách 2 vừa tối ưu vừa ngắn nhất; cách 3 và 4 chủ yếu để luyện mẫu.

## 12. Bộ test

Test 1: ví dụ đề và edge case (`1 <= len <= 100`, mọi `s, t` trong `1..100`).

In [14]:
impls = [SolutionLoop(), SolutionMin(), SolutionSort(), SolutionBinary(), Solution()]

def oracle(tasks):
    # mô phỏng thời gian: tăng T từ 0 cho tới khi có việc xong
    T = 0
    while True:
        for s, t in tasks:
            if s + t <= T:
                return T
        T += 1

tests = [
    ([[1, 6], [2, 3]], 5),
    ([[100, 100], [100, 100], [100, 100]], 200),
    ([[1, 1]], 2),                         # nhỏ nhất có thể
    ([[100, 100]], 200),                   # lớn nhất có thể
    ([[1, 100], [100, 1]], 101),           # hai việc cùng kết thúc lúc 101
    ([[1, 9], [5, 5], [9, 1]], 10),        # hai việc kết thúc lúc 10, một việc lúc 10
    ([[4, 2], [1, 9], [3, 1], [6, 6]], 4),
    ([[50, 50], [1, 50], [50, 1]], 51),    # bắt đầu sớm nhất không phải kết thúc sớm nhất
    ([[1, 50], [50, 50], [30, 1]], 31),
    ([[7, 3]] * 100, 10),                  # độ dài tối đa
]
for tasks, exp in tests:
    assert 1 <= len(tasks) <= 100 and all(len(p) == 2 and 1 <= p[0] <= 100 and 1 <= p[1] <= 100 for p in tasks)
    assert oracle(tasks) == exp, (tasks, oracle(tasks), exp)
    for im in impls:
        assert im.earliestTime([list(p) for p in tasks]) == exp, (type(im).__name__, tasks)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 10


Test 2: ngẫu nhiên có seed, đúng constraints (nhiều kích thước và nhiều khoảng giá trị).

In [15]:
import random
random.seed(3683)

for t in range(4000):
    n = random.randint(1, 100)
    hi = random.choice([1, 3, 10, 100])
    tasks = [[random.randint(1, hi), random.randint(1, hi)] for _ in range(n)]
    exp = oracle(tasks)
    for im in impls:
        assert im.earliestTime([list(p) for p in tasks]) == exp, (type(im).__name__, tasks)
print("4000 test ngẫu nhiên OK")

4000 test ngẫu nhiên OK


Test 3: tính chất — hoán vị công việc không đổi đáp án; thêm một công việc không bao giờ làm đáp án **lớn hơn**; đáp án nằm trong `[2, 200]`; và dịch tất cả `s` thêm `d` (nếu vẫn `<= 100`) tăng đáp án đúng `d`.

In [16]:
random.seed(683)
for _ in range(1500):
    n = random.randint(1, 99)
    tasks = [[random.randint(1, 50), random.randint(1, 100)] for _ in range(n)]
    base = Solution().earliestTime(tasks)
    assert 2 <= base <= 200
    shuffled = [list(p) for p in tasks]
    random.shuffle(shuffled)
    assert Solution().earliestTime(shuffled) == base
    extra = tasks + [[random.randint(1, 100), random.randint(1, 100)]]
    assert Solution().earliestTime(extra) <= base
    d = random.randint(1, 50)
    assert Solution().earliestTime([[s + d, t] for s, t in tasks]) == base + d
print("tính chất (hoán vị, thêm việc, biên, dịch thời gian): OK")

tính chất (hoán vị, thêm việc, biên, dịch thời gian): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: lấy `min` của thời điểm **bắt đầu**

In [17]:
def wrong_min_start(tasks):
    return min(s for s, t in tasks)                       # SAI: thiếu cộng thời lượng

print("sai :", wrong_min_start([[1, 6], [2, 3]]))
print("đúng:", Solution().earliestTime([[1, 6], [2, 3]]))
assert wrong_min_start([[1, 6], [2, 3]]) != Solution().earliestTime([[1, 6], [2, 3]])

sai : 1
đúng: 5


Vì sao sai: `1` là thời điểm **bắt đầu** sớm nhất, không phải lúc có việc **hoàn thành**. **Sửa:** `min(s + t)`.

### Lỗi 2: lấy `min` của thời lượng

In [18]:
def wrong_min_duration(tasks):
    return min(t for s, t in tasks)                       # SAI: bỏ qua thời điểm bắt đầu

print("sai :", wrong_min_duration([[1, 6], [2, 3]]))
print("đúng:", Solution().earliestTime([[1, 6], [2, 3]]))
assert wrong_min_duration([[1, 6], [2, 3]]) != Solution().earliestTime([[1, 6], [2, 3]])

sai : 3
đúng: 5


Vì sao sai: việc ngắn nhất (`t = 3`) bắt đầu lúc `2`, nên kết thúc lúc `5`, không phải `3`. **Sửa:** cộng `s` vào `t` trước khi `min`.

### Lỗi 3: cộng hai giá trị nhỏ nhất riêng rẽ `min(s) + min(t)`

In [19]:
def wrong_sum_of_mins(tasks):
    return min(s for s, t in tasks) + min(t for s, t in tasks)       # SAI: hai giá trị nhỏ nhất không thuộc cùng một việc

print("sai :", wrong_sum_of_mins([[1, 6], [2, 3]]))
print("đúng:", Solution().earliestTime([[1, 6], [2, 3]]))
assert wrong_sum_of_mins([[1, 6], [2, 3]]) != Solution().earliestTime([[1, 6], [2, 3]])

sai : 4
đúng: 5


Vì sao sai: `min(s) = 1` thuộc việc 1, `min(t) = 3` thuộc việc 2; ghép chúng cho `4` — một công việc **không tồn tại**. **Sửa:** cộng `s + t` **trong cùng một cặp** rồi mới lấy `min`.

### Lỗi 4: dùng `max` thay vì `min`

In [20]:
def wrong_max(tasks):
    return max(s + t for s, t in tasks)                   # SAI: đây là lúc TẤT CẢ việc đã xong

print("sai :", wrong_max([[1, 6], [2, 3]]))
print("đúng:", Solution().earliestTime([[1, 6], [2, 3]]))
assert wrong_max([[1, 6], [2, 3]]) != Solution().earliestTime([[1, 6], [2, 3]])

sai : 7
đúng: 5


Vì sao sai: `max(s + t)` là thời điểm mọi công việc xong (`7`); đề hỏi lúc **ít nhất một** việc xong (`5`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Sớm nhất ít nhất một việc xong" -> `min` trên **thời điểm kết thúc** của từng việc.
- "Mọi việc xong" -> `max`; "việc thứ `k` xong" -> sắp xếp rồi lấy phần tử thứ `k`.
- Luôn tính **cặp** `(s, t)` cùng nhau; đừng lấy `min` riêng từng cột.
- Công thức cơ bản: `kết thúc = bắt đầu + thời lượng`.
- Với giá trị nhỏ (`<= 100`), mọi cách đều đạt; ưu tiên cách ngắn nhất và ít lỗi.
- Nếu có ràng buộc "chỉ làm một việc một lúc" thì bài đổi bản chất (lập lịch), đọc kỹ đề.

## 15. Biến thể của bài

1. Thời điểm **tất cả** công việc xong (`max` của `s + t`).
2. Thời điểm công việc xong **thứ `k`** (sắp xếp hoặc heap).
3. Chỉ làm **một việc một lúc**, chọn thứ tự để **tổng thời gian** nhỏ nhất (bài lập lịch).
4. Cho các công việc có **hạn chót** `d`; đếm công việc xong trước hạn.
5. Trả về **chỉ số** công việc xong sớm nhất (khi hòa thì chỉ số nhỏ nhất).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp `best` | theo dõi min | `O(n)` | `O(1)` | hiểu thuật toán |
| 2. `min(s + t ...)` | một dòng | `O(n)` | `O(1)` | **nên nộp** |
| 3. Sắp xếp theo `s + t` | sắp rồi lấy đầu | `O(n log n)` | `O(n)` | cần thêm thứ tự xong |
| 4. Nhị phân theo đáp án | `feasible(T)` đơn điệu | `O(n log V)` | `O(1)` | luyện mẫu nhị phân |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().earliestTime([[3, 3], [1, 4]])` là gì?

**Bài 2 (tính tay):** `tasks = [[10, 5], [2, 20], [7, 3]]`. Thời điểm kết thúc từng việc và đáp án?

**Bài 3 (điền TODO):** hoàn thành `latest_time(tasks)` — thời điểm **tất cả** công việc đã xong.

**Bài 4 (tự viết lại):** viết `earliest_index(tasks)` trả về **chỉ số** của công việc xong sớm nhất (hòa thì chỉ số nhỏ nhất).

**Bài 5 (mini challenge):** viết `kth_finish(tasks, k)` — thời điểm công việc xong **thứ `k`** (`1 <= k <= len(tasks)`).

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().earliestTime([[3, 3], [1, 4]]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def latest_time(tasks):
    # TODO: max của s + t
    return -1

5


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** với mỗi việc tính `s + t`; sau đó chọn `min` / `max` / chỉ số / thứ `k`.

**Mức 2 (vừa):** Bài 3: `max(s + t for ...)`. Bài 4: `min(range(n), key=lambda i: tasks[i][0] + tasks[i][1])` (Python chọn chỉ số nhỏ nhất khi hòa). Bài 5: `sorted(s + t ...)[k - 1]`.

**Mức 3 (gần đáp án):** Bài 1: `3+3 = 6` và `1+4 = 5` -> `5`. Bài 2: `15, 22, 10` -> `10`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
assert Solution().earliestTime([[3, 3], [1, 4]]) == 5

# Bài 2
tasks2 = [[10, 5], [2, 20], [7, 3]]
assert [s + t for s, t in tasks2] == [15, 22, 10]
assert Solution().earliestTime(tasks2) == 10

# Bài 3
def latest_time(tasks):
    return max(s + t for s, t in tasks)
assert latest_time([[1, 6], [2, 3]]) == 7
assert latest_time([[100, 100]]) == 200

# Bài 4
def earliest_index(tasks):
    return min(range(len(tasks)), key=lambda i: tasks[i][0] + tasks[i][1])
assert earliest_index([[1, 6], [2, 3]]) == 1
assert earliest_index([[5, 5], [5, 5]]) == 0
assert earliest_index([[10, 5], [2, 20], [7, 3]]) == 2

# Bài 5
def kth_finish(tasks, k):
    return sorted(s + t for s, t in tasks)[k - 1]
assert kth_finish([[10, 5], [2, 20], [7, 3]], 1) == 10
assert kth_finish([[10, 5], [2, 20], [7, 3]], 2) == 15
assert kth_finish([[10, 5], [2, 20], [7, 3]], 3) == 22
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "thời điểm sớm nhất trong các thực thể độc lập":**

```python
return min(s + t for s, t in tasks)      # kết thúc = bắt đầu + thời lượng, lấy nhỏ nhất
```

**Checklist trước khi submit:**

- [ ] Tính **thời điểm kết thúc** `s + t` cho **từng** công việc (cùng một cặp)?
- [ ] Lấy `min` (ít nhất một xong), không phải `max` (tất cả xong)?
- [ ] Không lấy `min` riêng của cột bắt đầu hay cột thời lượng?
- [ ] Đã thử mảng một phần tử và trường hợp các việc hòa nhau (ví dụ 2)?
- [ ] Đã thử việc bắt đầu sớm nhất nhưng kết thúc muộn (`[[1,6],[2,3]]`)?
- [ ] Chữ ký đúng `earliestTime(self, tasks: List[List[int]]) -> int`?